# 09 · Coastal SST analysis from the satellite record
Source: **ESA CCI / C3S** satellite SST (`settings.yaml → analysis.coastal_source`), zone series from `scripts/build_satellite_validation.py`, 1993–2024. Coastal trends and upwelling are analysed here from observations, following the validation in notebook 08. Zones: 20°E split; shelf ≤ 200 m; nearshore ≤ 50 km from the coast.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()))
from src import utils, data_access, preprocessing, visualisation
settings = utils.load_settings()
utils.set_seed(settings['project']['random_seed'])
%matplotlib inline
from src import modelling, explainability
import pandas as pd, numpy as np, matplotlib.pyplot as plt
visualisation.apply_style()
df = data_access.load_zone_series(settings=settings)
print('source:', df.attrs['source'], df.index.min().date(), '→', df.index.max().date())
ZONES = ['benguela_nearshore', 'benguela_shelf', 'benguela_offshore', 'agulhas_nearshore', 'agulhas_shelf', 'agulhas_offshore']
df = df[ZONES]
LABEL = {z: z.replace('_', ' ').title() for z in ZONES}
COLORS = {'benguela_nearshore': '#0b2559', 'benguela_shelf': '#1e3a8a', 'benguela_offshore': '#60a5fa',
          'agulhas_nearshore': '#7c2d12', 'agulhas_shelf': '#f59e0b', 'agulhas_offshore': '#b91c1c'}
CREDIT = 'Data: ESA CCI/C3S SST via Copernicus Marine (SST_GLO_SST_L4_REP_OBSERVATIONS_010_024)'

## Mean state and seasonal cycle

In [ ]:
stats = pd.concat({z: preprocessing.describe_series(df[z])['value'] for z in ZONES}, axis=1)
stats.to_csv('../outputs/tables/09_sat_zone_stats.csv')
stats

In [ ]:
clims = pd.DataFrame({z: preprocessing.monthly_climatology(df[z]) for z in ZONES})
fig, ax = plt.subplots(figsize=(6.5, 3.4))
for z in ZONES:
    ax.plot(clims.index, clims[z], marker='o', ms=3, color=COLORS[z], label=LABEL[z])
ax.set(title='Satellite SST monthly climatology 1993–2024', xlabel='Month', ylabel='SST (°C)', xticks=range(1, 13))
ax.legend(fontsize=7, ncol=2); visualisation.source_note(ax, CREDIT); fig.tight_layout()
visualisation.save(fig, '09_sat_seasonality')

## Upwelling index: nearshore − offshore ΔT
More negative = colder nearshore relative to offshore (stronger upwelling signature). Thermal proxy only; wind data would be needed for attribution.

In [ ]:
dT = pd.DataFrame({r: df[f'{r}_nearshore'] - df[f'{r}_offshore'] for r in ('benguela', 'agulhas')})
dT.to_csv('../data/processed/sst_satellite_upwelling_index.csv')
dT.groupby(dT.index.month).mean().round(2).T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), gridspec_kw={'width_ratios': [1, 2.2]})
for r, c in (('benguela', '#1e3a8a'), ('agulhas', '#b91c1c')):
    m = dT[r].groupby(dT.index.month).mean(); axes[0].plot(m.index, m, marker='o', ms=3, color=c, label=r.title())
    axes[1].plot(dT[r], color=c, lw=0.5, alpha=0.35)
    axes[1].plot(dT[r].rolling(12, center=True).mean(), color=c, lw=1.6, label=f'{r.title()} (12-month mean)')
axes[0].set(title='Seasonal cycle of ΔT', xlabel='Month', ylabel='ΔT (°C)', xticks=range(1, 13))
axes[1].set(title='ΔT (nearshore − offshore), satellite', ylabel='ΔT (°C)')
axes[0].axhline(0, color='0.3', lw=0.8); axes[1].axhline(0, color='0.3', lw=0.8)
axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
visualisation.source_note(axes[1], CREDIT); fig.tight_layout(); visualisation.save(fig, '09_upwelling_index')

## Trends with false-discovery-rate control
Annual and seasonal trends for the six zones and both ΔT indices (40 tests). HAC 95 % CIs; significance after **Benjamini–Hochberg FDR** at 5 % across all 40.

In [ ]:
series = {**{z: df[z] for z in ZONES}, 'benguela_dT': dT['benguela'], 'agulhas_dT': dT['agulhas']}
rows = []
for name, s in series.items():
    a = preprocessing.anomalies(s)
    rows.append({'series': name, 'season': 'ANN', **preprocessing.hac_trend(a)})
    for season, r in preprocessing.seasonal_trends(a).iterrows():
        rows.append({'series': name, 'season': season, **r.to_dict()})
tr = pd.DataFrame(rows)
tr['significant_fdr'], tr['p_fdr'] = preprocessing.fdr_adjust(tr['p_value'])
tr = tr.set_index(['series', 'season'])
tr.round(4).to_csv('../outputs/tables/09_sat_trends_fdr.csv')
fmt = lambda r: f"{r.trend_degC_per_decade:+.2f} ± {r.ci95:.2f}{' **' if r.significant_fdr else (' *' if r.p_value < 0.05 else '')}"
print('** significant after FDR; * nominal p < 0.05 only')
tr.apply(fmt, axis=1).unstack('season')[['ANN', 'DJF', 'MAM', 'JJA', 'SON']].loc[list(series)]

In [ ]:
names = list(series); seasons = ['ANN', 'DJF', 'MAM', 'JJA', 'SON']
M = tr['trend_degC_per_decade'].unstack('season').loc[names, seasons]
S = tr['significant_fdr'].unstack('season').loc[names, seasons]
N = (tr['p_value'] < 0.05).unstack('season').loc[names, seasons]
fig, ax = plt.subplots(figsize=(6, 4.6))
im = ax.imshow(M.values, cmap='RdBu_r', vmin=-0.3, vmax=0.3, aspect='auto')
for i in range(len(names)):
    for j in range(len(seasons)):
        mark = '**' if S.iloc[i, j] else ('*' if N.iloc[i, j] else '')
        ax.text(j, i, f'{M.iloc[i, j]:+.2f}{mark}', ha='center', va='center', fontsize=7)
ax.set_xticks(range(len(seasons)), seasons)
ax.set_yticks(range(len(names)), [n.replace('_dT', ' ΔT').replace('_', ' ').title().replace('Δt', 'ΔT') for n in names], fontsize=8)
ax.axhline(5.5, color='k', lw=0.8); ax.grid(False)
fig.colorbar(im, ax=ax, label='°C per decade', shrink=0.8)
ax.set_title('Satellite SST trends 1993–2024\n** FDR-significant, * nominal p < 0.05', fontsize=10)
visualisation.source_note(ax, 'Data: ESA CCI/C3S SST via Copernicus Marine'); fig.tight_layout()
visualisation.save(fig, '09_sat_trend_matrix')

## One-month-ahead forecasting on satellite zone series
Chronological split from `settings.yaml` (train ≤ 2015, validation 2016–2019, test 2020–2024 **untouched**). Lags 1, 2, 3, 12 + month encoding.

In [ ]:
m = settings['modelling']; rows = {}; splits = {}
for z in ZONES:
    sp = modelling.chronological_split(modelling.make_features(df[z], m['lags']),
                                       m['train_end'], m['validation_end'], m['test_end'])
    splits[z] = sp
    rows[z] = modelling.evaluate(sp, settings['project']['random_seed'], on='validation')
skill = pd.concat(rows, names=['zone', 'model'])
skill.round(3).to_csv('../outputs/tables/09_sat_validation_metrics.csv')
print('Test period:', splits[ZONES[0]].test.index.min().date(), '→', splits[ZONES[0]].test.index.max().date(), '(not evaluated)')
skill['skill_vs_climatology'].unstack().round(3)[['seasonal_persistence', 'linear', 'random_forest', 'xgboost']]

## SHAP: Benguela nearshore
Explains how the best tree model uses its inputs, not what drives upwelling.

In [ ]:
z = 'benguela_nearshore'; sp = splits[z]; X = modelling.feature_cols(sp.train)
best = skill.loc[z].loc[['random_forest', 'xgboost'], 'RMSE'].idxmin()
model = modelling.build_models(settings['project']['random_seed'])[best]
model.fit(sp.train[X], sp.train['y'])
exp = explainability.shap_values(model, sp.validation[X])
imp = explainability.mean_abs_importance(exp, X)
imp.round(3).to_csv('../outputs/tables/09_shap_benguela_nearshore.csv')
print('model:', best); imp.round(3)

In [ ]:
fig = explainability.summary_plot(exp, sp.validation[X])
visualisation.save(plt.gcf(), '09_shap_benguela_nearshore')